# Module 2 — Building the End-to-End System with LangGraph

> **Time:** 25 minutes.
>
> **What you'll build:** the 4-agent Customer Support Triage system, end-to-end, runnable on a real ticket by the time you're done.

The design from Module 1 (state schema + topology) becomes runnable code in this notebook. We'll:

1. Install + set up
2. Define the typed `TriageState`
3. Build each agent — Classifier, Retriever, Drafter, QA
4. Wire them into a `StateGraph` with a feedback edge and a termination guard
5. Run it end-to-end on a billing ticket

**Definition of done:** `app.invoke(...)` returns a drafted reply for a sample ticket.

> *Solution notebook. Try the starter first.*

## 1.  Install + setup

Same dependencies as Module 0. If you already have them installed in this Colab session, the install is a no-op.

In [1]:
%pip install -q \
    langgraph==0.2.* \
    langchain==0.3.* \
    langchain-openai==0.2.* \
    pydantic==2.* \
    openai==1.*


import os

def _ensure_key(name: str, optional: bool = False) -> None:
    """Load an API key from (in order): existing env, Colab Secrets, or getpass.

    Colab Secrets are the recommended path for this workshop — set them ONCE
    via the 🔑 key icon in Colab's left sidebar and every notebook will pick
    them up automatically.
    """
    if os.environ.get(name):
        print(f"  ✓  {name} already set in environment")
        return
    # 1. Try Colab Secrets
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f"  ✓  {name} loaded from Colab Secrets")
            return
    except Exception:
        pass
    # 2. Fallback — prompt the user
    import getpass
    val = getpass.getpass(f"Paste your {name}{' (optional)' if optional else ''}: ").strip()
    if val:
        os.environ[name] = val
        print(f"  ✓  {name} set")
    elif optional:
        print(f"  •  {name} skipped (optional)")
    else:
        print(f"  ⚠   {name} skipped — you'll hit errors later without it")

_ensure_key("OPENAI_API_KEY")

print("Ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.
  ✓  OPENAI_API_KEY loaded from Colab Secrets
Ready.


## 2.  The knowledge base (inline)

In production this would be a vector store. For the workshop we use a small in-memory dict keyed by category — deterministic, free, fast.

Each article has `id`, `title`, `text`, and `tags`. The Retriever just looks up by category and returns the matching articles.

In [2]:
KB = {
    "billing": [
        {"id": "B-001",
         "title": "Billing cycle and prorations",
         "text": ("We bill on the same day each month. If you change plans mid-cycle, "
                  "the next invoice is prorated. Higher-than-usual charges are most "
                  "often a proration or a plan upgrade from the prior month.")},
        {"id": "B-002",
         "title": "Refund policy",
         "text": ("Refunds are available within 14 days of the original charge. "
                  "Refunds are issued to the original payment method and typically "
                  "settle within 5 business days.")},
        {"id": "B-003",
         "title": "Failed payments and retries",
         "text": ("If a payment fails, we retry once a day for 3 days. After that "
                  "the account is moved to a 7-day grace period.")},
    ],
    "technical": [
        {"id": "T-001",
         "title": "Login and authentication issues",
         "text": ("Most login failures resolve by clearing cookies for the domain. "
                  "If MFA isn't arriving, check spam and verify the registered phone.")},
        {"id": "T-002",
         "title": "API rate limits and 429 errors",
         "text": ("API rate limits are 60 requests per minute on Standard, 600/min "
                  "on Enterprise. 429 responses include a Retry-After header — "
                  "respect it with exponential backoff.")},
        {"id": "T-003",
         "title": "Data export failures",
         "text": ("Exports over 1 GB are split into multiple parts. If an export "
                  "shows 'failed', retry from the same dialog — the job is idempotent.")},
    ],
    "account": [
        {"id": "A-001",
         "title": "Resetting your password",
         "text": ("Use the 'Forgot password' link on the sign-in page. The reset "
                  "email is valid for 30 minutes. If you don't receive it, check "
                  "spam and confirm the address matches the registered account.")},
        {"id": "A-002",
         "title": "Closing or deleting an account",
         "text": ("Account closure is initiated from Settings → Account → Close "
                  "account. The account enters a 30-day pending-deletion window "
                  "during which it can be restored.")},
        {"id": "A-003",
         "title": "Transferring account ownership",
         "text": ("Ownership transfer requires the current owner to invite the new "
                  "owner as an Admin first, then both parties confirm via email.")},
    ],
}

# Quick sanity check
for cat, articles in KB.items():
    print(f"  {cat:10s}  {len(articles)} articles")

  billing     3 articles
  technical   3 articles
  account     3 articles


## 3.  The state schema

From Module 1's design. `revisions` uses the `Annotated[list, add]` reducer so each new revision is *appended* across loop iterations.

In [3]:
from typing import TypedDict, Annotated, Literal
from operator import add


class TriageState(TypedDict):
    ticket: str
    category: Literal["billing", "technical", "account"] | None
    urgency:  Literal["low", "med", "high"] | None
    retrieved: list[dict]
    draft: str
    verdict: Literal["pass", "revise"] | None
    revision_count: int
    revisions: Annotated[list[str], add]


def make_initial_state(ticket: str) -> TriageState:
    """Build a fresh state for a new ticket."""
    return {
        "ticket": ticket,
        "category": None,
        "urgency": None,
        "retrieved": [],
        "draft": "",
        "verdict": None,
        "revision_count": 0,
        "revisions": [],
    }


print(make_initial_state("Why is my bill so high?"))

{'ticket': 'Why is my bill so high?', 'category': None, 'urgency': None, 'retrieved': [], 'draft': '', 'verdict': None, 'revision_count': 0, 'revisions': []}


## 4.  LLM instance

We use **`gpt-4o-mini`** throughout — cheap, fast, plenty capable for this case study. `temperature=0` so reruns are reproducible (Module 3 will thank us for this).

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Smoke test
reply = llm.invoke("Reply 'pong' in lowercase.")
print(reply.content)

pong


## 5.  Agent 1 — Classifier

In [7]:
CLASSIFY_PROMPT = """\
You are a support ticket classifier.

Categorize the ticket into ONE of: billing, technical, account.
Rate urgency as ONE of: low, med, high.

High urgency = customer mentions a deadline, demo, outage, or repeated failures.
Medium = standard issue affecting work.
Low = informational question.

Return JSON only, no commentary:
{{"category": "...", "urgency": "..."}}

TICKET:
{ticket}
"""

In [8]:
import json

def parse_json(text: str) -> dict:
    """Tolerant JSON parser — handles models that occasionally wrap output in ```."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    return json.loads(text.strip())


def classify(state: TriageState) -> dict:
    """Read the ticket. Decide category + urgency. Return ONLY the keys this node sets."""
    prompt = CLASSIFY_PROMPT.format(ticket=state["ticket"])
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {
        "category": parsed["category"],
        "urgency":  parsed["urgency"],
    }


# Quick test
result = classify(make_initial_state("My bill jumped by $40 this month, please explain."))
print(result)

{'category': 'billing', 'urgency': 'low'}


## 6.  Agent 2 — Retriever

In [9]:
def retrieve(state: TriageState) -> dict:
    """Look up KB articles for the classified category."""
    category = state["category"]
    docs = KB.get(category, [])
    return {"retrieved": docs}


# Test
test_state = make_initial_state("My bill jumped this month.")
test_state["category"] = "billing"
print(retrieve(test_state))

{'retrieved': [{'id': 'B-001', 'title': 'Billing cycle and prorations', 'text': 'We bill on the same day each month. If you change plans mid-cycle, the next invoice is prorated. Higher-than-usual charges are most often a proration or a plan upgrade from the prior month.'}, {'id': 'B-002', 'title': 'Refund policy', 'text': 'Refunds are available within 14 days of the original charge. Refunds are issued to the original payment method and typically settle within 5 business days.'}, {'id': 'B-003', 'title': 'Failed payments and retries', 'text': 'If a payment fails, we retry once a day for 3 days. After that the account is moved to a 7-day grace period.'}]}


## 7.  Agent 3 — Drafter

In [10]:
DRAFT_PROMPT = """\
You are a customer support agent.

Reply to the ticket using ONLY the policies below. If a policy doesn't \
answer the question, say so honestly — do not invent details.

POLICIES:
{context}

TICKET:
{ticket}

Write a concise, helpful reply (3-6 sentences). Don't sign off with a name.
"""

In [11]:
def draft(state: TriageState) -> dict:
    """Write a draft reply grounded in retrieved policies."""
    context = "\n\n".join(
        f"[{d['id']}] {d['title']}\n{d['text']}"
        for d in state["retrieved"]
    )
    prompt = DRAFT_PROMPT.format(context=context, ticket=state["ticket"])
    response = llm.invoke(prompt)
    draft_text = response.content
    return {
        "draft": draft_text,
        "revisions": [draft_text],   # appended thanks to the reducer
    }


# Test
test_state = make_initial_state("My bill jumped by $40 this month, please explain.")
test_state["category"] = "billing"
test_state["retrieved"] = KB["billing"]
result = draft(test_state)
print(result["draft"])

Thank you for reaching out. A $40 increase in your bill could be due to a proration from changing plans mid-cycle or an upgrade from the prior month. We bill on the same day each month, so if there were any changes made, they would reflect in your next invoice. If you need further clarification on your specific charges, please check your billing details or let us know.


## 8.  Agent 4 — QA Reviewer

In [12]:
QA_PROMPT = """\
You are a quality assurance reviewer for a customer support team.

Decide whether the DRAFT below is acceptable to send to the customer.

Criteria for PASS:
- Addresses the ticket directly
- Uses information consistent with the policies provided
- Tone is professional and helpful

Otherwise, return REVISE.

Return JSON only:
{{"verdict": "pass" | "revise", "reason": "..."}}

TICKET:
{ticket}

POLICIES USED:
{context}

DRAFT:
{draft}
"""

In [13]:
def qa(state: TriageState) -> dict:
    """Review the draft. Decide pass or revise. Increment revision_count."""
    context = "\n\n".join(
        f"[{d['id']}] {d['title']}\n{d['text']}"
        for d in state["retrieved"]
    )
    prompt = QA_PROMPT.format(
        ticket=state["ticket"],
        context=context,
        draft=state["draft"],
    )
    response = llm.invoke(prompt)
    parsed = parse_json(response.content)
    return {
        "verdict": parsed["verdict"],
        "revision_count": state["revision_count"] + 1,
    }


# Test
test_state = make_initial_state("My bill jumped by $40, please explain.")
test_state.update({
    "category": "billing",
    "retrieved": KB["billing"],
    "draft": "Your bill is high because of prorations from a recent plan change.",
})
print(qa(test_state))

{'verdict': 'revise', 'revision_count': 1}


## 9.  Wire the graph

In [14]:
from langgraph.graph import StateGraph, END

MAX_REVISIONS = 2


def route_qa(state: TriageState) -> str:
    """Conditional edge after QA — pass to END, revise to drafter (bounded)."""
    if state["verdict"] == "pass":
        return "END"
    if state["revision_count"] >= MAX_REVISIONS:
        return "END"   # termination guard
    return "drafter"


# Build the graph
graph = StateGraph(TriageState)

graph.add_node("classify", classify)
graph.add_node("retrieve", retrieve)
graph.add_node("drafter",  draft)
graph.add_node("qa",       qa)

graph.set_entry_point("classify")

graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "drafter")
graph.add_edge("drafter",  "qa")

graph.add_conditional_edges(
    "qa",
    route_qa,
    {"drafter": "drafter", "END": END},
)

app = graph.compile()
print("Graph compiled.")

Graph compiled.


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 9.  Run the full pipeline

Invoke the graph on a sample ticket. The full state at the end shows what each node contributed.

In [15]:
result = app.invoke(make_initial_state(
    "Hi, my monthly bill is $50 higher than last month and I didn't change anything. "
    "Can you check what happened?"
))

print(f"category        :  {result['category']}")
print(f"urgency         :  {result['urgency']}")
print(f"retrieved       :  {[d['id'] for d in result['retrieved']]}")
print(f"verdict         :  {result['verdict']}")
print(f"revision_count  :  {result['revision_count']}")
print()
print("DRAFT:")
print(result["draft"])

category        :  billing
urgency         :  medium
retrieved       :  ['B-001', 'B-002', 'B-003']
verdict         :  revise
revision_count  :  2

DRAFT:
It sounds like the increase in your monthly bill may be due to a proration or a plan upgrade from the prior month, as we bill on the same day each month. Unfortunately, I cannot check specific account details, but I recommend reviewing your previous invoice for any changes that might explain the higher charge. If you have further questions or need assistance, please let us know.


## 10.  Watch each node fire — `stream()`

For debugging (Module 3 territory), it's useful to see each node's update as it happens. This is your "trace by hand" tool.

In [16]:
for step in app.stream(make_initial_state(
    "I want to close my account. Will everything really be deleted?"
)):
    for node, update in step.items():
        print(f"\n→  {node}")
        for k, v in update.items():
            preview = repr(v)
            if len(preview) > 80:
                preview = preview[:77] + "..."
            print(f"     {k} = {preview}")


→  classify
     category = 'account'
     urgency = 'low'

→  retrieve
     retrieved = [{'id': 'A-001', 'title': 'Resetting your password', 'text': "Use the 'Forgot...

→  drafter
     draft = 'To close your account, you can initiate the process from Settings → Account ...
     revisions = ['To close your account, you can initiate the process from Settings → Account...

→  qa
     verdict = 'pass'
     revision_count = 1


## Wrap up

You shipped your first multi-agent system today. Recap of what's now wired up:

- ✅  Typed `TriageState` with a reducer for revisions
- ✅  Four agents, each a function returning a partial state update
- ✅  Sequential pipeline with a conditional feedback edge
- ✅  Termination guard (`max_revisions = 2`) preventing infinite loops
- ✅  Both `invoke()` and `stream()` working end-to-end

**Up next:** Module 3 — Debugging & Failure Modes. We'll break this system on purpose and learn to spot what went wrong.